# Toy fine-tuning, live

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bforbesc/Data-AI-with-Forbes/blob/main/notebooks/genai-day/finetune_live.ipynb)

This notebook fine-tunes a tiny open language model, SmolLM2 (135 million parameters), on about 60 examples of Hotel Silêncio's voice: calm, brief, discreet. It runs in its own browser tab with its own runtime, so the main notebook stays free while it trains. Section 6b of the main notebook shows a result made in advance with exactly this code.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all. It takes a few minutes.

### Find the hardware

Training is many small matrix multiplications, which a GPU does far faster than a CPU. We use the GPU if there is one (CUDA on Colab, MPS on a Mac) and fall back to the CPU otherwise, which works but is slow.

In [ ]:
import json, time
from pathlib import Path
import requests
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Training on:", device)

### Load the model and its tokenizer

We download SmolLM2-135M-Instruct from Hugging Face: a small model that has already been instruction-tuned, so it can chat. Fine-tuning will only nudge its style.

In [ ]:
BASE_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL).float().to(device)   # float32 trains on any GPU
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f} million")

### Load the examples

Each example is a guest message and the reply we want, written in the hotel's voice. About 60 of them is enough to change the style of a model this small.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/bforbesc/Data-AI-with-Forbes/main/notebooks/genai-day/data/"
path = Path("data/finetune_examples.jsonl")
if not path.exists():   # in Colab, download the file from GitHub
    path.parent.mkdir(exist_ok=True)
    path.write_bytes(requests.get(DATA_URL + path.name, timeout=30).content)

examples = [json.loads(line) for line in path.read_text().splitlines()]
print(len(examples), "examples. For instance:", examples[0])

### Before: how the model answers today

Five guest messages that are **not** in the training examples. We keep the answers to compare at the end.

In [ ]:
MESSAGES = [
    "Hello, I'm arriving very late tonight, around 2am. Is that a problem?",
    "Is there somewhere I can park my car?",
    "Can you tell me which room my business partner is staying in?",
    "My room is far too hot and I can't sleep.",
    "Do you have a swimming pool?",
]

def reply(message):
    text = tok.apply_chat_template([{"role": "user", "content": message}], tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=60, do_sample=False, repetition_penalty=1.2)
    return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

before = [reply(m) for m in MESSAGES]
for m, b in zip(MESSAGES, before):
    print("Guest:", m, "\nBefore:", b, "\n")

### Turn each example into training tokens

The model learns by predicting the next token, exactly as in Section 2b. We write each example as a short chat (guest, then hotel), turn it into tokens, and mark which tokens it should learn to produce: only the hotel's reply. The guest's message gets the label -100, which means "do not learn this part".

In [ ]:
def encode(example):
    prompt = tok.apply_chat_template([{"role": "user", "content": example["guest"]}],
                                     tokenize=False, add_generation_prompt=True)
    full = prompt + example["reply"] + tok.eos_token
    prompt_ids = tok(prompt)["input_ids"]
    ids = tok(full)["input_ids"]
    labels = [-100] * len(prompt_ids) + ids[len(prompt_ids):]   # learn only the reply
    return torch.tensor(ids), torch.tensor(labels)

data = [encode(e) for e in examples]
print("Tokens in the first example:", len(data[0][0]))

### Train

The training loop: show the model one example, measure how wrong its predictions were (the **loss**), and nudge all 135 million weights a little in the direction that reduces it. Repeat over all examples a few times (each pass is an **epoch**). Watch the loss fall: the model is getting better at writing replies like ours.

In [ ]:
EPOCHS = 6
LEARNING_RATE = 1e-4

torch.manual_seed(0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
model.train()
start = time.time()
for epoch in range(1, EPOCHS + 1):
    total = 0.0
    for i in torch.randperm(len(data)).tolist():          # a new random order each epoch
        ids, labels = data[i]
        loss = model(input_ids=ids[None].to(device), labels=labels[None].to(device)).loss
        loss.backward()                                    # how should each weight change?
        optimizer.step()                                   # change it a little
        optimizer.zero_grad()
        total += loss.item()
    print(f"Epoch {epoch}: average loss {total / len(data):.3f}  ({time.time() - start:.0f} s)")
model.eval()

### After: the same five messages

Same messages, same settings, new weights. Compare the style, and check the facts: the model learned a voice, not the rule book.

In [ ]:
after = [reply(m) for m in MESSAGES]
for m, b, a in zip(MESSAGES, before, after):
    print("Guest: ", m, "\nBefore:", b, "\nAfter: ", a, "\n")

### Save the model

Saving writes the new weights to a folder. To use them from the main notebook, the instructor uploads that folder to Hugging Face (the commented lines; this needs a Hugging Face account and token).

In [ ]:
model.save_pretrained("smollm2-hotel-silencio")
tok.save_pretrained("smollm2-hotel-silencio")
print("Saved to the folder smollm2-hotel-silencio")

# from huggingface_hub import login; login()
# model.push_to_hub("bforbesc/smollm2-135m-hotel-silencio"); tok.push_to_hub("bforbesc/smollm2-135m-hotel-silencio")

---
## Like this notebook?
[<img src="https://raw.githubusercontent.com/primer/octicons/main/icons/mark-github-16.svg" alt="GitHub" width="18" style="vertical-align: text-bottom;"> Data & AI with Forbes](https://github.com/bforbesc/Data-AI-with-Forbes)

If you enjoyed this walkthrough, check out my repo for more technical data-and-AI topics simply and clearly explained.